- source_excel: stores the GT and model prediction values, and must contain the following columns
FMF_GT	FMF_Pred_dark	FMA_GT	FMA_Pred_dark	IFA_GT	IFA_Pred_dark	PL_Dist_GT	PL_Dist_Pred_dark
- save_report_xlsx: saves the final statistical analysis results as an Excel file
- save_image_png: saves the generated Bland-Altman plot as an image

In [ ]:
SOURCE_EXCEL=r""
SAVE_REPORT_XLSX=r""
SAVE_IMAGE_PNG=r""

In [7]:
import pandas as pd
import numpy as np

from scipy.stats import pearsonr
from scipy.stats import ttest_rel
from scipy.stats import sem
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

import pingouin as pg

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import pearsonr, ttest_rel, sem
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pingouin as pg
'''
SOURCE_EXCEL = os.environ.get('SOURCE_EXCEL')
SAVE_REPORT_XLSX = os.environ.get('SAVE_REPORT_XLSX')

if SOURCE_EXCEL is None:
    SOURCE_EXCEL = os.path.join(os.getcwd(), 'data', 'static.xlsx')
if SAVE_REPORT_XLSX is None:
    SAVE_REPORT_XLSX = os.path.join(os.getcwd(), 'output', 'clinical_4_groups_stat_report.xlsx')

'''
def clinical_analysis_for_batch(df, gt_col, pred_col, indicator_name):
    """
    Receive a loaded DataFrame, align valid paired samples, and perform statistical analysis.
    """
    gt_series = pd.to_numeric(df[gt_col], errors='coerce')
    pred_series = pd.to_numeric(df[pred_col], errors='coerce')

    paired_mask = gt_series.notna() & pred_series.notna()
    gt = gt_series[paired_mask].values
    pred = pred_series[paired_mask].values
    n = len(gt)

    if n < 3:
        print(f"Warning: [{indicator_name}] there are too few valid paired samples (N={n}); skipping analysis.")
        return None

    # =====================================
    # Error Metrics & Correlation
    # =====================================
    diff = pred - gt
    mae = mean_absolute_error(gt, pred)
    rmse = np.sqrt(mean_squared_error(gt, pred))
    bias = np.mean(diff)
    sd_diff = np.std(diff, ddof=1)

    loa_lower = bias - 1.96 * sd_diff
    loa_upper = bias + 1.96 * sd_diff

    r, p_corr = pearsonr(gt, pred)
    r2 = r2_score(gt, pred)
    t_stat, p_value = ttest_rel(gt, pred)

    def mean_ci(x):
        mean = np.mean(x)
        ci = 1.96 * sem(x)
        return mean, mean - ci, mean + ci

    gt_mean, gt_l, gt_u = mean_ci(gt)
    pred_mean, pred_l, pred_u = mean_ci(pred)
    # =====================================
    # ICC
    # =====================================
    icc_df = pd.DataFrame({
        'Target': np.repeat(np.arange(n), 2).astype(str),
        'Rater': ['Manual', 'AI'] * n,
        'Score': np.column_stack([gt, pred]).ravel().astype(float)
    })

    print(icc_df.head(10))
    icc = np.nan
    icc_ci = 'N/A'

    try:
        icc_result = pg.intraclass_corr(data=icc_df, targets='Target', raters='Rater', ratings='Score')
        type_col = 'Type' if 'Type' in icc_result.columns else 'type'

        if 'ICC(A,1)' in icc_result[type_col].values:
            icc_row = icc_result[icc_result[type_col] == 'ICC(A,1)'].iloc[0]
            icc = round(icc_row['ICC'], 3)
            ci_col = 'CI95' if 'CI95' in icc_result.columns else 'CI95%'
            ci_val = icc_row[ci_col]
            icc_ci = f"[{ci_val[0]:.3f}, {ci_val[1]:.3f}]"

        print('ICC result')
        print(icc_result)
    except Exception as e:
        print(f"Warning: [{indicator_name}] ICC calculation failed: {e}")

    return {
        'Clinical_Indicator': indicator_name,
        'Valid_Samples': n,
        'AI_Mean_Group': f"{pred_mean:.3f} ({pred_l:.3f}-{pred_u:.3f})",
        'Manual_Group': f"{gt_mean:.3f} ({gt_l:.3f}-{gt_u:.3f})",
        'Mean_Difference': f"{bias:.3f} ({loa_lower:.3f} to {loa_upper:.3f})",
        'MAE': round(mae, 3),
        'RMSE': round(rmse, 3),
        'R2_Score': round(r2, 3),
        'Pearson_r': round(r, 3),
        'Paired_t_test_p': f"{p_value:.6f}",
        'ICC_2': icc if not np.isnan(icc) else 'N/A',
        'ICC_95%_CI': icc_ci
    }


def batch_run_clinical_analysis(excel_path, output_report_path, sheet_name='Sheet1'):
    """Main control loop."""
    print(f"Reading source table: {excel_path}")
    df_total = pd.read_excel(excel_path, sheet_name=sheet_name)

    target_tasks = [
        {'indicator': 'FMF', 'gt': 'FMF_GT', 'pred': 'FMF_Pred_dark'},
        {'indicator': 'FMA', 'gt': 'FMA_GT', 'pred': 'FMA_Pred_dark'},
        {'indicator': 'IFA', 'gt': 'IFA_GT', 'pred': 'IFA_Pred_dark'},
        {'indicator': 'PL_Dist', 'gt': 'PL_Dist_GT', 'pred': 'PL_Dist_Pred_dark'},
    ]

    all_indicator_reports = []

    for task in target_tasks:
        if task['gt'] not in df_total.columns or task['pred'] not in df_total.columns:
            print(f"Missing required columns: {task['gt']} or {task['pred']}")
            continue

        report = clinical_analysis_for_batch(
            df=df_total,
            gt_col=task['gt'],
            pred_col=task['pred'],
            indicator_name=task['indicator']
        )
        if report is not None:
            all_indicator_reports.append(report)

    if all_indicator_reports:
        df_final_report = pd.DataFrame(all_indicator_reports)
        output_dir = os.path.dirname(output_report_path)
        if output_dir:
            os.makedirs(output_dir, exist_ok=True)
        df_final_report.to_excel(output_report_path, index=False)
        print(f"\nClinical analysis completed. Report saved to:\n➡️ {output_report_path}")
        print("\nSummary table:")
        print(df_final_report[['Clinical_Indicator', 'Valid_Samples', 'MAE', 'ICC_2']].to_string(index=False))
    else:
        print('No valid indicator report was generated.')


if __name__ == '__main__':
    if not os.path.exists(SOURCE_EXCEL):
        raise FileNotFoundError(
            f"Source Excel file not found: {SOURCE_EXCEL}. "
            'Set SOURCE_EXCEL environment variable or place the file in the expected location.'
        )

    batch_run_clinical_analysis(SOURCE_EXCEL, SAVE_REPORT_XLSX, sheet_name='Statistical_analysis')


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SOURCE_EXCEL = globals().get('SOURCE_EXCEL') or os.environ.get('SOURCE_EXCEL') or os.path.join(os.getcwd(), 'data', 'static.xlsx')
SAVE_IMAGE_PNG = globals().get('SAVE_IMAGE_PNG') or os.environ.get('SAVE_IMAGE_PNG') or os.path.join(os.getcwd(), 'output', 'bland_altman_4_groups.png')


def plot_bland_altman_for_batch(df, gt_col, pred_col, indicator_name, ax):
    """
    Receive a DataFrame, filter valid paired samples, and draw a Bland-Altman plot.
    """
    gt_series = pd.to_numeric(df[gt_col], errors='coerce')
    pred_series = pd.to_numeric(df[pred_col], errors='coerce')

    paired_mask = gt_series.notna() & pred_series.notna()
    gt = gt_series[paired_mask].values
    pred = pred_series[paired_mask].values
    n = len(gt)

    if n < 3:
        print(f"Warning: [{indicator_name}] there are too few valid paired samples (N={n}); skipping plotting.")
        ax.text(0.5, 0.5, f'{indicator_name}\nInsufficient Data (N={n})',
                ha='center', va='center', fontsize=12, color='red')
        return

    mean_val = (gt + pred) / 2.0
    diff_val = gt - pred

    md = np.mean(diff_val)
    sd = np.std(diff_val, ddof=1)
    upper_loa = md + 1.96 * sd
    lower_loa = md - 1.96 * sd

    ax.scatter(mean_val, diff_val, color='coral', alpha=0.6,
               edgecolors='coral', marker='o', facecolors='none', s=25)

    ax.axhline(md, color='steelblue', linestyle='-', linewidth=1.5)
    ax.axhline(upper_loa, color='brown', linestyle='--', linewidth=1.2)
    ax.axhline(lower_loa, color='brown', linestyle='--', linewidth=1.2)

    x_min, x_max = ax.get_xlim()
    text_x = x_max - (x_max - x_min) * 0.02

    ax.text(text_x, upper_loa, f'+1.96 SD\n{upper_loa:.2f}', ha='right', va='bottom', fontsize=10, color='black')
    ax.text(text_x, md, f'Mean\n{md:.2f}', ha='right', va='bottom', fontsize=10, color='black')
    ax.text(text_x, lower_loa, f'-1.96 SD\n{lower_loa:.2f}', ha='right', va='top', fontsize=10, color='black')

    ax.set_title(f"{indicator_name} (N={n})", fontsize=14, pad=10)
    ax.set_xlabel('Mean of AI and Manual', fontsize=11)
    ax.set_ylabel('Manual - AI', fontsize=11)
    ax.tick_params(direction='in', top=True, right=True, labelsize=10)
    ax.grid(False)


def batch_run_bland_altman_plots(excel_path, output_fig_path, sheet_name='Sheet1'):
    """Main plotting loop."""
    print(f"Reading source table: {excel_path}")
    df_total = pd.read_excel(excel_path, sheet_name=sheet_name)

    target_tasks = [
        {'indicator': 'FMF', 'gt': 'FMF_GT', 'pred': 'FMF_Pred_dark'},
        {'indicator': 'FMA', 'gt': 'FMA_GT', 'pred': 'FMA_Pred_dark'},
        {'indicator': 'IFA', 'gt': 'IFA_GT', 'pred': 'IFA_Pred_dark'},
        {'indicator': 'PL_Dist', 'gt': 'PL_Dist_GT', 'pred': 'PL_Dist_Pred_dark'},
    ]
    #Set up a 2x2 grid for Bland-Altman plots
    fig, axes = plt.subplots(2, 2, figsize=(14, 11))
    axes = axes.flatten()

    for i, task in enumerate(target_tasks):
        ax = axes[i]
        # Check if required columns exist
        if task['gt'] not in df_total.columns or task['pred'] not in df_total.columns:
            print(f"Missing required columns: {task['gt']} or {task['pred']}")
            ax.text(0.5, 0.5, f"Missing columns for {task['indicator']}",
                    ha='center', va='center', fontsize=12, color='red')
            continue

        plot_bland_altman_for_batch(
            df=df_total,
            gt_col=task['gt'],
            pred_col=task['pred'],
            indicator_name=task['indicator'],
            ax=ax
        )

    plt.tight_layout()

    # Save the figure to the specified path
    output_dir = os.path.dirname(output_fig_path)
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
    plt.savefig(output_fig_path, dpi=300)
    plt.show()
    print(f"\nBland-Altman plots completed. Figure saved to:\n➡️ {output_fig_path}")


if __name__ == '__main__':
    if not os.path.exists(SOURCE_EXCEL):
        raise FileNotFoundError(
            f"Source Excel file not found: {SOURCE_EXCEL}. "
            'Set SOURCE_EXCEL environment variable or place the file in the expected location.'
        )

    batch_run_bland_altman_plots(SOURCE_EXCEL, SAVE_IMAGE_PNG, sheet_name='Statistical_analysis')
